
## CREATING FACT TABLE

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable


#### Reading Silver Data

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`

In [0]:
df_silver = spark.sql('SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`')


#### Reading all the dims

In [0]:
df_dealer = spark.sql('SELECT * FROM cars_cata.gold.dim_dealer ')
df_branch = spark.sql('SELECT * FROM cars_cata.gold.dim_branch ')
df_date = spark.sql('SELECT * FROM cars_cata.gold.dim_date ')
df_model = spark.sql('SELECT * FROM cars_cata.gold.dim_model ')


#### Getting Keys to fact Table

In [0]:
df_fact = df_silver.join(df_dealer, df_silver.Dealer_ID == df_dealer.Dealer_ID, 'left').join(df_branch, df_silver.Branch_ID == df_branch.Branch_ID, 'left').join(df_date, df_silver.Date_ID == df_date.Date_ID, 'left').join(df_model, df_silver.Model_ID == df_model.Model_ID, 'left').select(df_silver['Revenue'], df_silver['Units_Sold'],df_silver['Revenue_per_unit'], df_branch['dim_branch_key'], df_dealer['dim_dealer_key'], df_date['dim_date_key'], df_model['dim_model_key'])


In [0]:
display(df_fact)


#### WRITING FACT TABLE

In [0]:
if spark.catalog.tableExists('cars_cata.gold.fact_sales'):
    delta_tbl = DeltaTable.forPath(spark,'abfss://gold@adlscarproject.dfs.core.windows.net/fact_sales' )
    delta_tbl.alias("trg").merge(df_fact.alias("src"),'trg.dim_date_key = src.dim_date_key AND trg.dim_branch_key = src.dim_branch_key AND trg.dim_dealer_key = src.dim_dealer_key AND trg.dim_model_key = src.dim_model_key')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
  df_fact.write.format('delta')\
      .mode('overwrite')\
      .option("path", 'abfss://gold@adlscarproject.dfs.core.windows.net/fact_sales')\
      .saveAsTable('cars_cata.gold.fact_sales')

In [0]:
%sql
SELECT * FROM cars_cata.gold.fact_sales